# 🎙️ Transcripción de audio con identificación de hablantes

Este notebook transcribe un audio a texto y además identifica **quién habla y en qué minuto** (diarización de hablantes).

Combina dos herramientas:
- **[Whisper](https://github.com/openai/whisper)** (OpenAI): transcribe el audio a texto con marcas de tiempo.
- **[pyannote.audio](https://github.com/pyannote/pyannote-audio)**: detecta los cambios de hablante ("diarización").

Al final obtendrás un resultado como:
```
[00:00 - 00:12] Hablante 1: Buenos días, ¿cómo estás?
[00:12 - 00:20] Hablante 2: Muy bien, gracias. ¿Y tú?
```

## Antes de empezar
1. En Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **GPU** (acelera mucho el proceso).
2. Necesitas una cuenta gratuita de [Hugging Face](https://huggingface.co/join) y un **token de acceso**:
   - Créalo en https://huggingface.co/settings/tokens (tipo "Read" es suficiente).
   - Acepta las condiciones de uso de estos dos modelos (son gratuitos, solo hay que aceptar):
     - https://huggingface.co/pyannote/speaker-diarization-3.1
     - https://huggingface.co/pyannote/segmentation-3.0
3. Ejecuta las celdas en orden, de arriba hacia abajo (▶️). El paso 1 reinicia automáticamente el entorno a la mitad; eso es normal, sigue con la celda siguiente cuando vuelva a conectar.

## 1. Instalar dependencias

Esta instalación actualiza `numpy`, que Colab ya tiene cargado en memoria. Si no se reinicia el entorno después de instalar, aparecen errores como `AttributeError: 'numpy.ufunc' object has no attribute '__module__'`. Por eso, la segunda celda reinicia el entorno automáticamente apenas termina la instalación: verás que la sesión se desconecta un instante y vuelve a conectar sola. Es normal, no hay que hacer nada.

In [ ]:
!pip install -q -U openai-whisper pyannote.audio
!apt-get -qq install -y ffmpeg > /dev/null
print("Instalación terminada. Reiniciando el entorno en la siguiente celda...")

In [ ]:
# Reinicia el entorno de ejecución para que la nueva versión de numpy se cargue limpia.
# La sesión se desconectará y reconectará sola: es el comportamiento esperado.
import os
os.kill(os.getpid(), 9)

## 2. Verificar la instalación

**Continúa desde aquí después de que el entorno se haya reconectado.** No vuelvas a ejecutar las dos celdas del paso 1.

In [ ]:
import importlib

importlib.import_module("whisper")
importlib.import_module("pyannote.audio")
print("✅ Dependencias instaladas y cargadas correctamente.")

## 3. Iniciar sesión en Hugging Face

Pega tu token cuando se te pida (no se mostrará en pantalla).

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Pega aquí tu token de Hugging Face: ")
login(token=hf_token)

## 4. Subir el archivo de audio

Puedes subir el audio directamente desde tu computadora, o montar tu Google Drive si el archivo ya está ahí.

In [ ]:
# Opción A: subir el audio desde tu computadora
from google.colab import files

subido = files.upload()
AUDIO_PATH = next(iter(subido))
print(f"Archivo cargado: {AUDIO_PATH}")

In [ ]:
# Opción B (alternativa): usar un audio guardado en Google Drive.
# Descomenta las siguientes líneas si prefieres esta opción en vez de la anterior.

# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO_PATH = "/content/drive/MyDrive/ruta/a/tu/audio.mp3"

## 5. Configuración

- `WHISPER_MODEL`: tamaño del modelo de transcripción. De menor a mayor precisión (y más lento): `tiny`, `base`, `small`, `medium`, `large-v3`.
- `IDIOMA`: código del idioma del audio (`"es"` para español, `"en"` para inglés, etc.). Usa `None` para detección automática.
- `NUM_HABLANTES`: si sabes cuántas personas hablan en el audio, indícalo aquí para mejorar la precisión. Si no lo sabes, deja `None`.

In [ ]:
WHISPER_MODEL = "medium"
IDIOMA = "es"
NUM_HABLANTES = None  # por ejemplo: 2

## 6. Transcribir el audio con Whisper

Esto convierte el audio en texto, dividido en segmentos con su tiempo de inicio y fin.

In [ ]:
import whisper

print("Cargando modelo Whisper...")
modelo_whisper = whisper.load_model(WHISPER_MODEL)

print("Transcribiendo audio (puede tardar varios minutos)...")
resultado = modelo_whisper.transcribe(AUDIO_PATH, language=IDIOMA, verbose=False)

segmentos_transcripcion = resultado["segments"]
print(f"\nTranscripción completa: {len(segmentos_transcripcion)} segmentos detectados.")

## 7. Identificar a los hablantes (diarización)

Esto detecta en qué momentos del audio habla cada persona, sin importar todavía qué dicen.

In [ ]:
import torch
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

print("Cargando modelo de diarización...")
pipeline_diarizacion = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", token=hf_token
)

usa_gpu = torch.cuda.is_available()
if usa_gpu:
    pipeline_diarizacion.to(torch.device("cuda"))
print(f"Dispositivo: {'GPU ✅' if usa_gpu else 'CPU ⚠️ (será mucho más lento; revisa Entorno de ejecución → Cambiar tipo de entorno de ejecución)'}")

print("Analizando hablantes (la barra de progreso muestra que sigue avanzando)...")
parametros = {}
if NUM_HABLANTES:
    parametros["num_speakers"] = NUM_HABLANTES

with ProgressHook() as hook:
    diarizacion = pipeline_diarizacion(AUDIO_PATH, hook=hook, **parametros)

turnos_hablantes = [
    {"inicio": turno.start, "fin": turno.end, "hablante": hablante}
    for turno, _, hablante in diarizacion.itertracks(yield_label=True)
]

speakers_detectados = sorted(set(t["hablante"] for t in turnos_hablantes))
print(f"Hablantes detectados: {speakers_detectados}")

## 8. Combinar transcripción y hablantes

Cada fragmento de texto se asigna al hablante que más tiempo se solapa con él.

In [ ]:
def hablante_para_segmento(inicio_seg, fin_seg, turnos):
    mejor_hablante = None
    mejor_solape = 0.0
    for turno in turnos:
        solape = min(fin_seg, turno["fin"]) - max(inicio_seg, turno["inicio"])
        if solape > mejor_solape:
            mejor_solape = solape
            mejor_hablante = turno["hablante"]
    return mejor_hablante or "Desconocido"

segmentos_con_hablante = []
for seg in segmentos_transcripcion:
    hablante = hablante_para_segmento(seg["start"], seg["end"], turnos_hablantes)
    segmentos_con_hablante.append({
        "inicio": seg["start"],
        "fin": seg["end"],
        "hablante": hablante,
        "texto": seg["text"].strip(),
    })

# Agrupar segmentos consecutivos del mismo hablante en un solo bloque
bloques = []
for seg in segmentos_con_hablante:
    if bloques and bloques[-1]["hablante"] == seg["hablante"]:
        bloques[-1]["fin"] = seg["fin"]
        bloques[-1]["texto"] += " " + seg["texto"]
    else:
        bloques.append(dict(seg))

print(f"{len(bloques)} bloques de conversación generados.")

## 9. (Opcional) Poner nombres reales a los hablantes

Por defecto los hablantes se llaman `SPEAKER_00`, `SPEAKER_01`, etc. Si sabes quién es quién, complétalo aquí. Si no, deja el diccionario vacío y se usará "Hablante 1", "Hablante 2"...

In [ ]:
# Ejemplo: NOMBRES_HABLANTES = {"SPEAKER_00": "Ana", "SPEAKER_01": "Luis"}
NOMBRES_HABLANTES = {}

if not NOMBRES_HABLANTES:
    etiquetas_unicas = sorted(set(b["hablante"] for b in bloques))
    NOMBRES_HABLANTES = {
        etiqueta: f"Hablante {i + 1}" for i, etiqueta in enumerate(etiquetas_unicas)
    }

print(NOMBRES_HABLANTES)

## 10. Ver el resultado final

In [ ]:
def formatear_tiempo(segundos):
    segundos = int(segundos)
    h, resto = divmod(segundos, 3600)
    m, s = divmod(resto, 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

lineas_transcripcion = []
for b in bloques:
    nombre = NOMBRES_HABLANTES.get(b["hablante"], b["hablante"])
    linea = f"[{formatear_tiempo(b['inicio'])} - {formatear_tiempo(b['fin'])}] {nombre}: {b['texto']}"
    lineas_transcripcion.append(linea)

transcripcion_final = "\n".join(lineas_transcripcion)
print(transcripcion_final)

## 11. Guardar y descargar el resultado

Se generan dos archivos:
- `transcripcion.txt`: texto plano con hablante y minuto.
- `transcripcion.srt`: subtítulos, útil para reproducir el audio/video con el texto sincronizado.

In [ ]:
nombre_base = "transcripcion"

# Archivo de texto
with open(f"{nombre_base}.txt", "w", encoding="utf-8") as f:
    f.write(transcripcion_final)

# Archivo de subtítulos SRT
def formatear_tiempo_srt(segundos):
    h, resto = divmod(segundos, 3600)
    m, s = divmod(resto, 60)
    ms = int((s - int(s)) * 1000)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{ms:03d}"

with open(f"{nombre_base}.srt", "w", encoding="utf-8") as f:
    for i, b in enumerate(bloques, start=1):
        nombre = NOMBRES_HABLANTES.get(b["hablante"], b["hablante"])
        f.write(f"{i}\n")
        f.write(f"{formatear_tiempo_srt(b['inicio'])} --> {formatear_tiempo_srt(b['fin'])}\n")
        f.write(f"{nombre}: {b['texto']}\n\n")

print("Archivos generados. Descargando...")
from google.colab import files
files.download(f"{nombre_base}.txt")
files.download(f"{nombre_base}.srt")

## Solución de problemas

- **`ModuleNotFoundError: No module named 'whisper'`** (u otro módulo): la celda de instalación (paso 1) no se ejecutó en este entorno, o el entorno se reinició y perdió lo instalado. Usa **Entorno de ejecución → Reiniciar sesión** y vuelve a ejecutar desde el paso 1.
- **`AttributeError: 'numpy.ufunc' object has no attribute '__module__'`** (u otros errores raros justo después de instalar): el entorno no se reinició tras actualizar `numpy`. Ve a **Entorno de ejecución → Reiniciar sesión** manualmente y continúa desde el paso 2 (verificación), sin volver a ejecutar el paso 1.
- **`TypeError: Pipeline.from_pretrained() got an unexpected keyword argument 'use_auth_token'`**: tu versión de `pyannote.audio`/`huggingface_hub` ya no acepta ese nombre; el notebook usa `token=hf_token`, que es el actual. Si ves este error es que tienes una copia antigua del notebook: descarga la versión más reciente.
- **Error 401 / "gated model"**: no aceptaste las condiciones de los modelos de pyannote. Visita los dos enlaces de la sección de requisitos, inicia sesión y pulsa "Agree", luego vuelve a ejecutar el notebook.
- **Se queda sin memoria (OOM)**: usa un modelo de Whisper más pequeño (`small` o `base`) en el paso 5.
- **La diarización (paso 7) tarda muchísimo o parece congelada**: la celda ahora imprime si está usando `GPU ✅` o `CPU ⚠️`; si dice CPU, ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona GPU, luego reinicia y vuelve a ejecutar desde el paso 2. La barra de progreso (`ProgressHook`) te confirma que sigue avanzando aunque tarde. Como referencia, en GPU la diarización suele tardar entre el 5% y el 20% de la duración del audio (ej. un audio de 1 hora, unos 3-12 minutos); en CPU puede tardar más que la duración del propio audio.
- **Los hablantes se confunden**: si sabes cuántas personas hablan, indícalo en `NUM_HABLANTES` (paso 5).